# 055 — Introduction to Recurrent Neural Networks

A third architecture for a third kind of data. The numbers are stark: on a
10,000-word vocabulary padded to 100 words, an ANN's first layer needs
**128,000,128** parameters against an RNN's **1,296,512** — and the RNN's count
**does not change** when the sequences get longer.

| Part | What we check |
|---|---|
| A | the test for whether order matters |
| B | ANN against RNN parameters, and the 99x gap |
| C | the RNN count is independent of sequence length |
| D | how much of a padded tensor is padding, on real documents |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Part A — Does shuffling destroy the meaning?

That is the whole test for whether you need a sequence model.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
sentence = "the film was not good at all".split()
shuffled = list(rng.permutation(sentence))
print("original :", " ".join(sentence))
print("shuffled :", " ".join(shuffled))
print()

row = {"age": 34, "income": 52000, "city": "Chennai"}     # tabular
print("a tabular row shuffled is the same row:", list(rng.permutation(list(row))))
print()
print("Shuffling the sentence destroyed it; shuffling the column order did not.")
print("Text, time series, audio and DNA fail the test. Tabular rows pass it.")
assert sentence != shuffled

## Part B — What an ANN would cost on padded one-hot text

In [ ]:
VOCAB, MAXLEN, HIDDEN = 10_000, 100, 128

ann = MAXLEN * VOCAB * HIDDEN + HIDDEN
rnn = (VOCAB + HIDDEN) * HIDDEN + HIDDEN
print(f"vocabulary {VOCAB:,}, padded to {MAXLEN} words, {HIDDEN} hidden units")
print(f"ANN flattened input {MAXLEN * VOCAB:>14,} values")
print(f"ANN first layer     {ann:>14,} parameters")
print(f"RNN recurrent layer {rnn:>14,} parameters   ({ann / rnn:.0f}x fewer)")
assert ann == 128_000_128 and rnn == 1_296_512
assert round(ann / rnn) == 99
print()
print("Three separate problems, one fix: the ANN has no notion of order, needs")
print("a fixed input length, and pays for every position separately.")

## Part C — The count that does not grow

In [ ]:
print(f"  {'steps':>7} {'ANN first layer':>18} {'RNN layer':>12}")
for steps in (10, 100, 1_000, 10_000):
    print(f"  {steps:>7} {steps * VOCAB * HIDDEN + HIDDEN:>18,} {rnn:>12,}")
counts = {steps: (VOCAB + HIDDEN) * HIDDEN + HIDDEN for steps in (10, 10_000)}
assert counts[10] == counts[10_000] == rnn
print()
print("A thousand times longer sequences, the same recurrent layer. That is")
print("weight sharing across time, and it is the entire economy of an RNN.")

## Part D — How much of a padded tensor is zeros

On real documents, padding to the longest one wastes almost everything.

In [ ]:
try:
    from sklearn.datasets import fetch_20newsgroups
    docs = fetch_20newsgroups(subset="train",
                              remove=("headers", "footers", "quotes")).data
    lengths = np.array([len(t.split()) for t in docs])
    lengths = lengths[lengths > 0]
    print(f"{len(lengths):,} documents: median {int(np.median(lengths))} words, "
          f"mean {lengths.mean():.0f}, max {lengths.max():,}")
    for cap in (int(np.percentile(lengths, 95)), int(lengths.max())):
        used = np.minimum(lengths, cap).sum()
        waste = 1 - used / (cap * len(lengths))
        print(f"  padded to {cap:>6}: {waste:.0%} padding zeros")
        assert 0 < waste < 1
    cap95 = int(np.percentile(lengths, 95))
    assert 1 - np.minimum(lengths, cap95).sum() / (cap95 * len(lengths)) > 0.6
    print()
    print("Padding to the 95th percentile already makes most of the tensor")
    print("zeros. Padding to the maximum is far worse - truncate instead.")
except Exception as exc:
    print(f"skipped - corpus unavailable offline ({type(exc).__name__})")

## What this notebook established

- The test for a sequence model: does shuffling the inputs destroy the meaning?
- ANN first layer **128,000,128** parameters against an RNN's **1,296,512** —
  **99x** fewer, on the same vocabulary and length.
- The RNN count is **independent of sequence length**; the ANN's grows linearly.
- On real documents, padding to the 95th-percentile length leaves most of the
  input tensor as zeros.

## Exercises

1. At what `MAXLEN` does the ANN first layer pass a billion parameters?
2. Recompute the RNN count for `HIDDEN=256`. Which term dominates, and why?
3. Truncate the documents at the median length instead. How much padding is
   left, and what have you thrown away?